# Suite EXP — Expressions

An expression is an `Expressions.Framework.Expression`, a reference object whose `root` is a term: a value of
`Expressions.Framework.Term`, the union of the dialects' `Term`s: Patterns' `Term`, `Expressions.Framework.Patterns.Term`, lists
the framework's kinds `literal`, `call`, `variable` and `let`, the container constructors `list` and `mapping`, the
kinds of invalid syntax `error` and `missing`, and its own `quantifier`, and its install adds it to `Term` by schema
extension. A term carries no domain. `Framework.install` and `Patterns.install` define them in a store; terms are built there,
in a transaction, through mbse-schemas' builders (or Patterns' `Writer`s, which write through them), and read through its
readers, so they serialize, validate and compare like any other value. `Evaluators` computes them (suite EVL).

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.Framework import Patterns
from mbse.Expressions.Framework.Patterns import Evaluators as V, Expressions as E
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import Builders, Comparison as C, JSON, Plain, Validators, YAML
from support import raises

store = Builders.Store()
store.transaction(Framework.install)
store.transaction(Patterns.install)
Term = store.registered("Expressions.Framework.Term")
Expression = store.registered("Expressions.Framework.Expression")


def term(spec):
    """The term value `spec` writes, built in a transaction."""
    with store.transaction():
        return store.Expressions.Framework.Term(E.DIALECT.spec(spec))


lit = term(E.literal(3))
op = term(E.operation("f", 1, "x", lit))

<a id="EXP-01"></a>
## EXP-01 · Literals hold any value, with their spelling and its problems where they were read from source

In [ ]:
assert F.kind_of(lit)[0] == "literal" and F.form(lit) == F.Form("literal", {"value": 3}) and E.DIALECT.validate(lit) == []
assert lit.patterns.literal.value.get() == 3 and lit.as_schema() is Term  # read through the readers, its dialect first
assert F.dialect_of(lit) == "Expressions.Framework.Patterns.Term" and F.dialect_of(lit.patterns) == F.dialect_of(lit)
for value in [0, 2**70, -0.0, math.nan, "", b"", False]:  # every basic scalar
    assert E.DIALECT.validate(term(value)) == [] and F.same(term(value), term(E.literal(value)))
assert type(F.form(term(-0.0)).attributes["value"]) is float and math.copysign(1.0, term(-0.0).patterns.literal.value.get()) == -1.0
assert E.DIALECT.validate(term(E.literal(None))) == ["a literal needs a value"]
assert E.DIALECT.validate(term(lambda u: u.literal(lambda r: r))) == ["a literal needs a value"]
assert F.same(term(lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar("x")))), term("x"))  # the builders' own calls
# value is an unrestricted OfAny: a literal holds any value, a term value among them (CAP-233)
held = term(lambda u: u.literal(lambda r: r.value(lit)))
assert E.DIALECT.validate(held) == [] and F.same(F.form(held).attributes["value"], lit)
# a domain representation (PRI-023): its spelling, unvalidated, and a problem for each thing wrong with it
spelled = term(lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar(16)).text("0x10")))
assert F.form(spelled).attributes == {"value": 16, "text": "0x10"} and E.DIALECT.validate(spelled) == []
assert not F.same(spelled, term(16))  # the spelling is part of the term
invalid = term(lambda u: u.literal(lambda r: r.text("08").problems(["leading zeros", "not octal"])))  # no value
assert E.DIALECT.validate(invalid) == ["leading zeros", "not octal"]
with raises(ValueError, match="a literal needs a value"):
    V.OfAny(invalid)

<a id="EXP-02"></a>
## EXP-02 · Calls have a callable and ordered arguments

In [ ]:
form = F.form(op)
assert form.kind == "call" and form.attributes == {"callable": "f"} and [a.patterns.literal.value.get() for a in form.arguments] == [1, "x", 3]
assert F.same(form.arguments[2], lit) and op.patterns.call.callable.get() == "f"
assert E.DIALECT.validate(op) == ["'f' is not an operation Patterns defines"]  # written, and refused: Patterns defines no f
nested = term(lambda u: u.call(lambda c: c.callable("not").arguments([E.DIALECT.spec(op)])))
assert F.same(F.form(nested).arguments[0], op)
assert E.DIALECT.validate(term(lambda u: u.call(lambda c: c))) == ["a call needs a callable"]
assert E.DIALECT.validate(term(lambda u: u.call(lambda c: c.callable("")))) == ["a call needs a callable"]
assert E.DIALECT.validate(term(lambda u: u.call(lambda c: c.callable(3)))) == ["a call's callable must be a str, got int"]
bad = term(E.operation("f", E.literal(None), lambda u: u.call(lambda c: c.arguments([E.literal(None)]))))
assert E.DIALECT.validate(bad) == [
    "'f' is not an operation Patterns defines",
    "argument 0: a literal needs a value",
    "argument 1: a call needs a callable",
    "argument 1: argument 0: a literal needs a value",
]

<a id="EXP-03"></a>
## EXP-03 · A term is a nested value: written into another it is copied, so it holds no cycle

In [ ]:
pair = term(E.operation("g", op, op))  # each argument a copy of op
assert all(F.same(a, op) for a in F.form(pair).arguments)
assert len(list(F.walk(pair))) == 1 + 2 * len(list(F.walk(op)))  # nothing is shared: every term is visited
again = term(pair)
assert F.same(again, pair) and again is not pair

<a id="EXP-04"></a>
## EXP-04 · Terms are built through mbse-schemas' builders, in a transaction: a term alone, or an expression's root

In [ ]:
x = E.variable("x")
with store.transaction():
    source = store.Expressions.Framework.Expression(lambda e: e.root(E.operation("f", 1)))
assert Expression.ref and F.form(F.term(source)).attributes == {"callable": "f"}
with store.transaction():
    updated = store.Expressions.Framework.Expression(source, lambda e: e.root(E.operation("g", 2)))  # an update
    copy = store.Expressions.Framework.Expression(source)  # a clone
assert updated is source and F.form(F.term(source)).attributes == {"callable": "g"}
assert copy is not source and F.same(copy, source)
with raises(ValueError, match="no term is set"):
    with store.transaction():
        F.term(store.Expressions.Framework.Expression(lambda e: e))
assert F.same(F.term(source.root), source)  # an expression, its root property, or the term itself
with raises(TypeError, match="not a term: 3"):
    F.term(3)
# A spec is a writer, a term value (copied), a native value (a literal) or a callable on the term value's builder.
assert isinstance(E.DIALECT.spec(x), E.Writer) and isinstance(E.DIALECT.spec(lit), F.Writer)
assert isinstance(E.DIALECT.spec(1), E.Writer) and isinstance(E.DIALECT.spec(lambda u: u.variable(lambda v: v.name("x"))), F.Writer)
with raises(TypeError, match="expected a term, a writer or a native value, got <class 'int'>"):
    E.DIALECT.spec(int)
with raises(TypeError, match="expected a term, a writer or a native value, got [1]"):
    E.DIALECT.spec([1])
with raises(TypeError, match="expected a term, a writer or a native value, got None"):
    F.spec(None)
with raises(TypeError, match="expected a term, a writer or a native value, got 1"):
    F.spec(1, lambda value: None)  # a dialect with no literal for it

<a id="EXP-05"></a>
## EXP-05 · The framework's schemas are an install: ordinary, registered schemas, which Patterns' install extends

In [ ]:
before = JSON.ToJSON(store).Snapshot()
store.transaction(Framework.install)
store.transaction(Patterns.install)
assert JSON.ToJSON(store).Snapshot() == before  # run again, each install writes nothing
names = ["Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
         "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping",
         "Expressions.Framework.Patterns.Quantifier", "Expressions.Framework.Error", "Expressions.Framework.Missing"]
kinds = [store.registered(name) for name in names]
Pair = store.registered("Expressions.Framework.Pair")
Union = store.registered("Expressions.Framework.Patterns.Term")  # Patterns' union of its kinds
schemas = [Term, Union, Expression, store.registered("Expressions.Framework.HasRoot"), Pair, *kinds]
# the union of the dialects' `Term`s: each dialect's install adds its own, named after it; with Patterns alone, Term
# is a union of one branch, which validates, as do the schemas holding it and the expressions (mbse-schemas 0.26.6)
assert [b.name for b in Term.branches] == ["patterns"] and Term.branches[0].schema is Union
assert all(schema.validate() == [] for schema in schemas)
with store.transaction():
    alone = store.Expressions.Framework.Expression(lambda e: e.root(E.literal(1)))
assert Validators.Validate(store)(Expression, alone) == []
assert [b.name for b in Union.branches] == ["literal", "call", "variable", "let", "list", "mapping", "quantifier",
                                            "error", "missing"]
assert [b.schema for b in Union.branches] == kinds and not any(kind.ref for kind in kinds)  # value objects
assert [list(k.properties) for k in kinds] == [["value", "text", "problems"], ["callable", "arguments"], ["name"],
                                               ["name", "value", "body"], ["elements"], ["entries"],
                                               ["name", "quantifier", "collection", "body"], ["text", "terms"], ["kind"]]
assert kinds[4].properties["elements"].schema.element is Term and kinds[5].properties["entries"].schema.element is Pair
assert list(Pair.properties) == ["key", "value"] and all(p.schema is Term for p in Pair.properties.values())
assert kinds[1].properties["arguments"].schema.element is Term and kinds[3].properties["body"].schema is Term
assert kinds[6].properties["body"].schema is Union  # a dialect's own kind holds its dialect's terms; a shared one, any
assert kinds[7].properties["terms"].schema.element is Term
assert Expression.ref and list(Expression.properties) == ["root"] and Expression.properties["root"].schema is Term
Dialect = store.registered("Expressions.Framework.Dialect")  # a kind union: each dialect adds its singleton's schema
assert [b.name for b in Dialect.branches] == ["patterns"] and Dialect.branches[0].schema is store.registered("Expressions.Framework.Patterns.Dialect")
assert Dialect.branches[0].schema.ref and Dialect.branches[0].schema.singleton == "Expressions.Framework.Patterns"
HasRoot = store.registered("Expressions.Framework.HasRoot")
assert list(HasRoot.entry.links) == ["schema", "expression"] and HasRoot.uniques == (frozenset({"expression"}),)
OfExpression = store.registered("Schemas.OfExpression")
assert OfExpression.adjacencies["expression"].relation is HasRoot and OfExpression.adjacencies["expression"].me == "schema"

<a id="EXP-06"></a>
## EXP-06 · Round trips through Plain, JSON and YAML: a term written nested, its kind its branch's name

In [ ]:
expr = term(E.list_(E.operation("eq", 1, 1.5), "s", b"\x01", E.mapping((True, 1), ("k", E.variable("v")))))
expected = JSON.ToJSON(store)(Term, expr)
for encode, decode in [(Plain.ToPlain(store), Plain.FromPlain(store)), (JSON.ToJSON(store), JSON.FromJSON(store)),
                       (YAML.ToYAML(store), YAML.FromYAML(store))]:
    back = decode(Term, encode(Term, expr))
    assert back is not expr and F.same(back, expr) and E.DIALECT.validate(back, ["v"]) == []
    assert JSON.ToJSON(store)(Term, back) == expected
assert JSON.ToJSON(store)(Term, term(E.operation("not", True))) == (  # each term its dialect, then its kind
    '{"patterns": {"call": {"callable": "not", "arguments": [{"patterns": {"literal": {"value": {"bool": true}}}}]}}}')
alone = JSON.ToJSON(store)(Term, term(-0.0))
assert alone == '{"patterns": {"literal": {"value": {"float": -0.0}}}}'
assert math.copysign(1.0, JSON.FromJSON(store)(Term, alone).patterns.literal.value.get()) == -1.0
# A literal's value is an unrestricted OfAny's: written with its schema, one level (mbse-basis CAP-071).
for value, wire in [(1, '{"int": 1}'), (1.5, '{"float": 1.5}'), (True, '{"bool": true}'), ("x", '{"str": "x"}'),
                    (b"\x01", '{"bytes": "AQ=="}'), (math.nan, '{"float": "NaN"}')]:
    written = JSON.ToJSON(store)(Term, term(value))
    assert written == '{"patterns": {"literal": {"value": %s}}}' % wire
    assert F.same(JSON.FromJSON(store)(Term, written), term(value))
# nothing set: only the branches are written
assert [JSON.ToJSON(store)(Term, term(lambda u, k=k: getattr(u, k)(lambda r: r)))
        for k in ("literal", "call", "variable", "let", "list", "mapping", "error", "missing")] == [
    '{"patterns": {"%s": {}}}' % k for k in ("literal", "call", "variable", "let", "list", "mapping", "error", "missing")]
assert JSON.ToJSON(store)(Term, term(E.mapping(("a", 1)))) == (
    '{"patterns": {"mapping": {"entries": [{"key": {"patterns": {"literal": {"value": {"str": "a"}}}}, '
    '"value": {"patterns": {"literal": {"value": {"int": 1}}}}}]}}}')
with raises(Exception, match="has no branch 'operation'"):  # a dialect, then its kind
    JSON.FromJSON(store)(Term, '{"operation": {}}')
with raises(Exception, match="patterns.operation: the union has no branch 'operation'"):
    JSON.FromJSON(store)(Term, '{"patterns": {"operation": {}}}')
# A store holding expressions and Patterns' dialect, its domains (a packed one among them) and operations, round-trips
# whole, read into a store with the same installs: nothing lost, nothing written twice.
defined = store.transaction(Patterns.define)[0]
with store.transaction():
    store.Expressions.Framework.Expression(lambda e: e.root(expr))
    state = store.Expressions.Framework.Domain(lambda d: d.kind(lambda k: k.as_value(lambda a: a.as_union(
        lambda u: u.enum(lambda e: e.members(["IDLE", "RUN"]))))))
    mode = store.Expressions.Framework.Domain(lambda d: d.kind(lambda k: k.as_value(lambda a: a.as_union(lambda u: u.packed(
        lambda e: e.codes([0, 1]))))).packs(lambda a: a.insert(lambda e: e.domain(state).representation(defined["uint8"]))))
    store.Expressions.Framework.Patterns.Dialect(defined["dialect"], lambda b: b.domains({"state": lambda e: e.domain(state),
                                                                             "mode": lambda e: e.domain(mode)}))
snapshot = JSON.ToJSON(store).Snapshot()
again = Builders.Store()
again.transaction(Framework.install)
again.transaction(Patterns.install)
JSON.FromJSON(again).Snapshot(snapshot)
assert JSON.ToJSON(again).Snapshot() == snapshot
restored = again.Instances.Expressions.Framework.Patterns  # by its path
assert sorted(str(e.name) for e in restored.domains) == sorted(
    ["bool", "int", "float", "str", "bytes", "uint8", "binary32", "std_logic", "state", "mode"])
assert len(list(restored.operations)) == len(E.CORE)
packed = restored.domains({"name": "mode"})  # owned: found by its path, its dialect's then its name
assert Patterns.Domains.read(packed).name() == "packed(enum(IDLE, RUN) as uint8)" and Patterns.Domains.read(packed).validate() == []

<a id="EXP-07"></a>
## EXP-07 · An expression is a reference object; a schema's expression is linked by `HasRoot`, an extension adjacency

In [ ]:
with store.transaction():
    adult = store.Expressions.Framework.Expression(lambda e: e.root(E.variable("person").age.ge(18)))
    schema = store.Schemas.OfExpression(lambda s: s.name("exp.Adult").result([store.Schemas.OfBool])
                                        .expression(lambda a: a.insert(lambda e: e.expression(adult))))
linked = [entry.expression for entry in Builders.OfExpression.Builder(schema, False, store).expression]
assert len(linked) == 1 and F.same(linked[0], adult)
assert Validators.Validate(store)(Expression, adult) == []
assert V.OfAny(adult, {"person": None}) is None
# Patterns' own holder refines the framework's Expression, its root narrowed to Patterns' Term: HasRoot links it, and
# it is read, validated and evaluated as any expression.
Holder = store.registered("Expressions.Framework.Patterns.Expression")
with store.transaction():
    senior = store.Expressions.Framework.Patterns.Expression(lambda e: e.root(E.variable("person").age.ge(65)))
    elder = store.Schemas.OfExpression(lambda s: s.name("exp.Senior").result([store.Schemas.OfBool])
                                       .expression(lambda a: a.insert(lambda e: e.expression(senior))))
assert Holder.refines is Expression and F.is_expression(Holder) and not F.is_expression(Term)
assert [F.same(e.expression, senior) for e in Builders.OfExpression.Builder(elder, False, store).expression] == [True]
assert Validators.Validate(store)(Holder, senior) == [] and E.DIALECT.validate(senior, ["person"]) == []
assert F.dialect_of(F.term(senior)) == E.DIALECT.union and V.OfAny(senior, {"person": None}) is None

<a id="EXP-08"></a>
## EXP-08 · Validation and comparison work through the schemas

In [ ]:
assert Validators.Validate(store)(Expression, adult) == [] and E.DIALECT.validate(adult, bound={"person"}) == []


def compare(a, b):
    return C.OfUnion(Term, term(a)).compare(C.OfUnion(Term, term(b)))


assert compare(1, 1) == 0 and compare(1, 2) is None  # values are not ordered
assert compare(1, 1.0) is None  # another scalar
assert compare(E.operation("f", 1), E.operation("f", 1)) == 0 and compare(E.operation("f", 1), E.operation("f", 2)) is None
assert compare(E.variable("x"), E.literal("x")) is None  # another branch

<a id="EXP-09"></a>
## EXP-09 · Terms read and write through the visitor protocols

In [ ]:
names = [p.as_schema().name for p in op.patterns.call.as_properties]
assert names == ["callable", "arguments"]  # the properties set, schema unaware
entries = []
op.patterns.call.arguments.as_value(lambda a: a.as_container(lambda c: c.as_entries(entries.extend)))
assert [i for i, _ in entries] == [0, 1, 2]
values = []
for _, element in entries:
    element.as_union(lambda u: values.append(u.patterns.literal.value.get()))  # its dialect, then its kind
assert values == [1, "x", 3] and F.same(F.term(entries[0][1]), term(1)) and not F.is_term(entries[0][1])  # an element's reader holds a term
seen = []
term(lambda u: (seen.append(type(u.as_schema()).__name__), u.literal(lambda r: (seen.append(r.as_schema().name), r)[1]))[1])
assert seen == ["_UnionData", "Expressions.Framework.Literal"]  # a dialect's spec is given its dialect's `Term`
# A writer writes through what it is given: a term value's union, a property, a container's element; where it is given
# a value of Term, it selects its dialect's branch first, as a callable written through the builders does.
with store.transaction():
    written = store.Expressions.Framework.Expression(lambda e: e.root(lambda p: p.as_value(lambda a: a.as_union(
        lambda u: u.patterns(lambda d: d.call(lambda c: c.callable("f").arguments([
            E.literal(1), lambda y: y.as_union(lambda t: t.patterns(E.variable("x").write))])))))))
assert F.same(written, term(E.operation("f", 1, E.variable("x"))))

<a id="EXP-10"></a>
## EXP-10 · Variables and lets

In [ ]:
v = term(E.variable("x"))
assert F.name_of(v) == "x" and F.kind_of(v)[0] == "variable"
assert E.DIALECT.validate(v) == ["variable 'x' is not bound"] and E.DIALECT.validate(v, bound={"x"}) == []
assert E.DIALECT.validate(term(lambda u: u.variable(lambda r: r))) == ["a variable needs a name"]
assert E.DIALECT.validate(term(lambda u: u.variable(lambda r: r.name(1)))) == ["a variable's name must be a str, got int"]
let = term(E.let_("x", 1, E.variable("x")))
form = F.form(let)
assert form.attributes == {"name": "x"} and F.same(form.arguments[0], term(1)) and F.same(form.arguments[1], v)
assert E.DIALECT.validate(let) == [] and F.name_of(let) == "x"
assert E.DIALECT.validate(term(E.let_("x", None, None))) == ["a let needs a value", "a let needs a body"]
assert E.DIALECT.validate(term(lambda u: u.let(lambda r: r.value(E.literal(1)).body(E.variable("x"))))) == [
    "a let needs a name", "body: variable 'x' is not bound"]
same_name = term(E.let_("y", E.variable("y"), E.variable("y")))
assert E.DIALECT.validate(same_name) == ["value: variable 'y' is not bound"]  # the name is bound in the body only
open_let = term(E.let_("x", None, None))
assert F.form(open_let).arguments == (None, None) and F.name_of(term(E.literal(1))) is None
assert F.same(term(E.DIALECT.make(F.form(open_let))), open_let)

<a id="EXP-11"></a>
## EXP-11 · Writers build terms with methods

In [ ]:
person = E.variable("person")
writer = person.age.ge(18)
assert isinstance(writer, E.Writer) and callable(writer)
form = F.form(term(writer))
assert form.attributes == {"callable": "ge"} and form.arguments[1].patterns.literal.value.get() == 18
got = F.form(form.arguments[0])
assert got.attributes == {"callable": "get"} and F.name_of(got.arguments[0]) == "person" and got.arguments[1].patterns.literal.value.get() == "age"
assert F.form(term(person.get("eq"))).arguments[1].patterns.literal.value.get() == "eq"  # a property named like a method
methods = [person.eq(1), person.ne(1), person.lt(1), person.le(1), person.gt(1), person.ge(1), person.and_(True), person.or_(True),
           person.not_(), person.implies(True), person.add(1), person.sub(1), person.mul(1), person.neg(), person.has("x"),
           person.get("x")]
assert [F.form(term(m)).attributes["callable"] for m in methods] == [
    "eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not", "implies", "add", "sub", "mul", "neg", "has", "get"]
assert term(E.literal(3)).patterns.literal.value.get() == 3 and F.form(term(E.operation("in", 1, 2))).attributes == {"callable": "in"}
bound = term(E.let_("a", 1, E.variable("a")))
assert F.name_of(bound) == "a" and F.same(term(bound), bound)
with raises(AttributeError):
    person._private  # Python's own probes are not properties

<a id="EXP-13"></a>
## EXP-13 · Dialects, operations and domains are data: a dialect's singleton names its operations and owns its domains

In [ ]:
defined = store.transaction(Patterns.define)[0]  # Patterns' dialect's data; run again, it writes nothing
assert store.transaction(Patterns.define)[0]["uint8"] is defined["uint8"]
basic = defined["dialect"]
assert basic is store.Instances.Expressions.Framework.Patterns is store.singleton("Expressions.Framework.Patterns")  # its path: its global name
Defines, Operates = store.registered("Expressions.Framework.Defines"), store.registered("Expressions.Framework.Operates")
assert list(Defines.entry.links) == ["dialect", "domain"] and list(Defines.entry.properties) == ["name"]
assert Defines.uniques == (frozenset({"dialect", "name"}), frozenset({"domain"}))
assert list(Operates.entry.links) == ["dialect", "operation"] and list(Operates.entry.properties) == ["name"]
assert Operates.uniques == (frozenset({"dialect", "name"}), frozenset({"operation"}))
Domain = store.registered("Expressions.Framework.Domain")
assert Domain.ref and Domain.adjacencies["dialect"].owner and Domain.adjacencies["dialect"].relation is Defines
# Its operations: expression schemas its module defines, named by the dialect; their variables, in order, its arguments.
operations = {str(entry.name): entry.operation for entry in basic.operations}
assert list(operations) == list(E.CORE) == list(Patterns.Domains.SIGNATURES)  # the vocabulary validation reads
assert {name: len(o.pattern.variables) for name, o in operations.items()} == E.CORE
assert all(o is store.registered(f"Expressions.Framework.Patterns.{name}") and o.validate() == [] for name, o in operations.items())
add, number = operations["add"], store.registered("Expressions.Framework.Patterns.Number")
assert list(add.pattern.variables) == ["a", "b"] and add.pattern.variables["a"].schema is number and add.result is number
assert list(operations["shl"].pattern.variables) == ["value", "count"]  # the order declared: arguments are positional
assert list(operations["in"].pattern.variables) == ["value", "collection"]
assert operations["in"].pattern.variables["collection"].schema is store.registered("Expressions.Framework.Patterns.Collection")
assert operations["eq"].result.schemas == (store.registered("Schemas.OfBool"),)  # a result is an OfAny
assert operations["get"].result.schemas == () and operations["not"].pattern.variables["a"].schema is store.registered("Schemas.OfBool")
restored = Builders.Store()  # the order survives a snapshot
JSON.FromJSON(restored).Snapshot(JSON.ToJSON(store).Snapshot())
assert list(restored.registered("Expressions.Framework.Patterns.shl").pattern.variables) == ["value", "count"]
assert E.DIALECT.validate(term(E.operation("contains", 1, 2))) == ["'contains' is not an operation Patterns defines"]
assert E.DIALECT.validate(term(E.operation("add", 1))) == ["add takes 2 arguments, got 1"]
assert "convert" not in E.CORE and "pack" not in E.CORE  # deferred with domain inference
assert basic.operations({"name": "add"}) is operations["add"]  # by its path, its dialect's then its name
# Its domains: owned by name, each related to its schema, an OfAny of its scalar, both ways.
domains = {str(entry.name): entry.domain for entry in basic.domains}
assert list(domains) == ["bool", "int", "float", "str", "bytes", "uint8", "binary32", "std_logic", "state", "mode"]  # EXP-06's too
assert domains["uint8"] is defined["uint8"] is basic.domains({"name": "uint8"})
uint8 = store.registered("Expressions.Framework.Patterns.uint8")
wrap8 = Patterns.Domains.OfInteger.Builder().width(8).signed(False).overflow("wrap").create()
assert uint8.schemas == (store.registered("Schemas.OfInt"),) and [e.schema for e in domains["uint8"].schema] == [uint8]
assert [Patterns.Domains.read(e.domain) for e in Builders.OfAny.Builder(uint8, False, store).domain] == [wrap8]  # its schema's
assert Patterns.Domains.read(domains["uint8"]) == wrap8
assert all(Patterns.Domains.read(d).validate() == [] for d in domains.values())
assert JSON.ToJSON(store)(Domain, domains["uint8"]).count('"wrap"') == 1  # a document of its own, by its path